# 04_problem_image_detect

원본 Colab 셀에서 분리. (`#@title 문제 이미지 검출`)


In [ ]:
#@title [매 세션 1] rfdetr 설치
#@markdown 런타임이 끊기면 설치된 패키지가 전부 사라지므로, 매 세션 rfdetr을 다시 설치해야 함
!pip install -q "rfdetr[train,loggers]"                  # RF-DETR 학습(train)+로깅(loggers) 의존성

In [ ]:
#@title [매 세션 2] 드라이브 마운트 + 경로 자동 탐색
#@markdown 세션마다 드라이브 연결이 끊기므로 재마운트 필요. 데이터·zip이 전부 드라이브에 있어 경로부터 다시 잡아야 함
from google.colab import drive                           # 코랩↔드라이브 연결 도구
drive.mount('/content/drive')                             # 드라이브 마운트

import os, glob                                          # 경로·탐색 도구
CANDS = [                                                # 흔한 후보 경로 먼저
    '/content/drive/MyDrive/1팀 공유 문서/ai12-level1-project/sprint_ai_project1_data',
    '/content/drive/MyDrive/ai12-level1-project/sprint_ai_project1_data',
]
DATA_ROOT = next((c for c in CANDS if os.path.exists(c)), None)   # 존재하는 첫 경로 채택
if DATA_ROOT is None:                                    # 후보에 없으면 전체 재귀 검색
    hits = glob.glob('/content/drive/MyDrive/**/sprint_ai_project1_data', recursive=True)
    DATA_ROOT = hits[0] if hits else None
assert DATA_ROOT, "sprint_ai_project1_data를 못 찾음"     # 못 찾으면 중단
PROJ_ROOT = os.path.dirname(DATA_ROOT)                   # zip·백업 공통 상위 경로

TEST_IMG = os.path.join(DATA_ROOT, 'test_images')        # 제출용 842장(추론 때 사용)
print("DATA_ROOT:", DATA_ROOT)                           # 채택 경로 확인
print("PROJ_ROOT:", PROJ_ROOT)

In [ ]:
#@title [매 세션 3] dataset_74 zip 복원
import os                                                     # 경로 확인 도구

ZIP = os.path.join(PROJ_ROOT, "dataset_74_5fold.zip")         # 74클래스 5-fold zip(드라이브)
print("zip 존재:", os.path.exists(ZIP))                        # True 확인

!cp "$ZIP" /content/                                           # 드라이브 → 로컬 복사(속도 위해)
!unzip -qo /content/dataset_74_5fold.zip -d /content/dataset_74  # 압축 해제(-o=덮어쓰기)
print("복원 fold:", sorted(d for d in os.listdir("/content/dataset_74") if d.startswith("fold")))

In [ ]:
#@title 문제 이미지 검출
import json, os                                   # json 읽기, 경로 조합용
from PIL import Image, ImageDraw                  # 이미지 열기/크롭, 박스 그리기
import matplotlib.pyplot as plt                   # 그리드 출력
import math                                       # 행 수 계산용

# ===== 설정 =====
ANNO_SOURCES = [                                  # fold0~4 train/valid 전부 (중복은 코드가 제거)
    ("/content/dataset_74/fold0/train/_annotations.coco.json", "/content/dataset_74/fold0/train"),
    ("/content/dataset_74/fold0/valid/_annotations.coco.json", "/content/dataset_74/fold0/valid"),
    ("/content/dataset_74/fold1/train/_annotations.coco.json", "/content/dataset_74/fold1/train"),
    ("/content/dataset_74/fold1/valid/_annotations.coco.json", "/content/dataset_74/fold1/valid"),
    ("/content/dataset_74/fold2/train/_annotations.coco.json", "/content/dataset_74/fold2/train"),
    ("/content/dataset_74/fold2/valid/_annotations.coco.json", "/content/dataset_74/fold2/valid"),
    ("/content/dataset_74/fold3/train/_annotations.coco.json", "/content/dataset_74/fold3/train"),
    ("/content/dataset_74/fold3/valid/_annotations.coco.json", "/content/dataset_74/fold3/valid"),
    ("/content/dataset_74/fold4/train/_annotations.coco.json", "/content/dataset_74/fold4/train"),
    ("/content/dataset_74/fold4/valid/_annotations.coco.json", "/content/dataset_74/fold4/valid"),
]
LABEL_MAP_PATH = "/content/drive/MyDrive/1팀 공유 문서/김태윤/label_map_74.json"

TARGET_IDX = 73        # 0~73, 검수할 클래스 #55 68
PAD_RATIO = 0.2          # 박스 여백 비율
NCOLS = 5                # 그리드 한 줄당 이미지 수


def collect_unique_instances(sources, target_category_id):
    seen_keys = {}                                # (파일명, bbox튜플) -> 대표 인스턴스 저장 #물리적 박스 1개당 1번만
    for json_path, img_dir in sources:            # fold0~4 전체 순회
        with open(json_path, "r", encoding="utf-8") as f:
            coco = json.load(f)
        id2file = {img["id"]: img["file_name"] for img in coco["images"]}  # image_id -> 파일명

        for ann in coco["annotations"]:
            if ann["category_id"] != target_category_id:
                continue                          # 대상 클래스 아니면 skip
            fname = id2file[ann["image_id"]]
            key = (fname, tuple(ann["bbox"]))     # 파일명+bbox = 물리적으로 같은 박스 판별 키
            if key in seen_keys:
                continue                          # 다른 fold에서 이미 잡힌 같은 박스면 skip (중복 제거 핵심)
            img_path = os.path.join(img_dir, fname)  # 처음 만난 fold의 실제 이미지 경로를 대표로
            seen_keys[key] = (img_path, ann["bbox"], ann["id"], fname)

    return list(seen_keys.values())               # 중복 없는 고유 박스 목록


def crop_with_box(img_path, bbox, pad_ratio):
    img = Image.open(img_path).convert("RGB")
    x, y, w, h = bbox
    pad_x, pad_y = w * pad_ratio, h * pad_ratio

    left   = max(0, int(x - pad_x))
    top    = max(0, int(y - pad_y))
    right  = min(img.width,  int(x + w + pad_x))
    bottom = min(img.height, int(y + h + pad_y))

    crop = img.crop((left, top, right, bottom))
    draw = ImageDraw.Draw(crop)
    box_in_crop = [x - left, y - top, x - left + w, y - top + h]
    draw.rectangle(box_in_crop, outline="red", width=3)
    return crop


def is_valid_bbox(img_path, bbox, pad_ratio):
    # 좌표 역전(손상) 사전 체크하는 도구
    img = Image.open(img_path)
    x, y, w, h = bbox
    pad_x, pad_y = w * pad_ratio, h * pad_ratio
    left   = max(0, int(x - pad_x))
    top    = max(0, int(y - pad_y))
    right  = min(img.width,  int(x + w + pad_x))
    bottom = min(img.height, int(y + h + pad_y))
    return (right > left) and (bottom > top)


def show_grid(crops, labels, ncols):
    n = len(crops)
    if n == 0:
        print("해당 클래스 인스턴스 없음 (annotation 0)")
        return
    nrows = math.ceil(n / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3.2, nrows * 3.6))  # 파일명 길어 폭 넓힘
    axes = axes.flatten() if n > 1 else [axes]
    for i, ax in enumerate(axes):
        if i < n:
            ax.imshow(crops[i])
            ax.set_title(labels[i], fontsize=5)   # 전체 파일명이라 폰트 작게
        ax.axis("off")
    plt.tight_layout()
    plt.show()


# ===== 실행 =====
target_category_id = TARGET_IDX + 1

with open(LABEL_MAP_PATH, "r", encoding="utf-8") as f:
    label_map = json.load(f)
print(f"TARGET_IDX={TARGET_IDX} -> category_id={target_category_id}")
print("label_map 참고 항목:", label_map.get(str(target_category_id), "키 형식 확인 필요"))

instances = collect_unique_instances(ANNO_SOURCES, target_category_id)  # 중복 제거된 고유 박스
print(f"고유 박스(알약 사진) 수: {len(instances)}")

good, bad = [], []                                 # 정상/손상 분리
for p, b, aid, fname in instances:
    (good if is_valid_bbox(p, b, PAD_RATIO) else bad).append((p, b, aid, fname))

print(f"정상: {len(good)} / 손상: {len(bad)}")
if bad:
    print("\n=== 손상된 bbox (좌표 역전) ===")
    for p, b, aid, fname in bad:
        print(f"file_name={fname} | ann_id={aid} | bbox={b}")

crops = [crop_with_box(p, b, PAD_RATIO) for p, b, _, _ in good]
labels = [f"{fname}\nann_id={aid}" for _, _, aid, fname in good]  # 전체 파일명 + ann_id (자르지 않음)

show_grid(crops, labels, NCOLS)